# Section 2 — roll timing starter

**Lucie: change roll timing only.** Destination, capacity handling, rounding, target weights, benchmark, accounting and metric stay fixed. Use the same outgoing contract and baseline next-deferred destination across all four timing rules.

Version 8 §§45–48 defines fixed -5 (baseline), fixed -10, lagged-volume crossover and lagged-OI crossover. The live roadmap checked on 11 October has no Part 2 timing-engine issue; confirm/create that open issue before your graded PR. [Issue #7](https://github.com/JacopoMichelacci/Futures_Class/issues/7) is the later Part 3 analysis, not the engine task.

Run all cells from a fresh Python kernel using the project's existing uv environment. These notebooks use only the shared synthetic fixture and standard-library Python, with no online access. Outputs are deliberately cleared before sharing.

**Part 2 responsibility:** each teammate implements their own reusable engine and substantive tests, opens their own issue-linked PR, and reviews teammate PRs. This starter supplies inputs and interface examples, not the strand solution. Keep engine logic in `src/` and tests in `tests/unit/`. No empirical conclusions or preferred-method selection belong here. Do not consume another strand's preferred results.

See [Section 0 interfaces](../docs/section0_phase_a.md) and [fixture provenance](../data/fixtures/README.md). Phase B still needs sourced first-notice metadata, verified exchange/shortened-session times, historical benchmark/baseline quantities, production tables, the common 100-date/70% gate and AUM/capacity preflight. The synthetic calendar is not a real CBOT calendar. Daily return Tracking RMSE in basis points remains the sole ranking metric for the later historical comparison.

In [ ]:
from pathlib import Path
from decimal import Decimal
from pprint import pprint
import sys

# Start the kernel in this repository or the notebook's own folder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "config/project.yaml").is_file()
             and (p / "data/fixtures/phase_a.json").is_file()
             and (p / "src/__init__.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the kernel in the repository or this notebook's folder.")
sys.path.insert(0, str(ROOT))
import src
assert Path(src.__file__).resolve().parent == ROOT / "src", "Restart the kernel for this checkout."
from src.validation.schemas import load_project_config, HOLDINGS_COLUMNS
from src.data.fixtures import load_fixture

config = load_project_config(ROOT / "config/project.yaml")
f = load_fixture(ROOT / "data/fixtures/phase_a.json",
                 config["max_liquidity_source_age_sessions"])
return_date = f.raw["return_dates"][0]
context = f.calendar.decision(return_date)
print("SYNTHETIC FIXTURES ONLY", config["roots"], "return interval ends", return_date)


### Baseline pair and observed-session roll map

Use explicit calendar indices. The generated event date is the decision session, and incoming holdings apply to the next interval. Later settlement publication times do not move the trading date. The synthetic shortened session illustrates an input requirement; real shortened-session verification remains Phase B.

In [ ]:
from src.portfolio.baseline import build_baseline_path
case = f.raw["baseline_path_case"]
path = build_baseline_path(case["eligible_curve"], case["decision_dates"],
                           case["commodities"], f.calendar, f.catalogue)
event = path.roll_events[0]
pprint(event)
pprint(path.positions[:3])


In [ ]:
from src.contracts.maturity import days_to_anchor
event_index = f.calendar.indices[event["date"]]
next_return_date = f.calendar.dates[event_index + 1]
event_context = f.calendar.decision(next_return_date)
assert days_to_anchor(f.calendar, event["date"], event["anchor"]) == -config["baseline_roll_offset_trading_days"]
pprint({"decision_date": event["date"], "cutoff": event_context.cutoff_utc,
        "incoming_interval_end": next_return_date, "outgoing": event["outgoing_contract"],
        "incoming": event["baseline_destination"], "anchor": event["anchor"]})
pprint([f.calendar.sessions[d].__dict__ for d in f.calendar.dates[event_index:event_index + 2]])


### Aligned volume/OI known at the cut-off

`latest_common` selects observations sharing one source trading date across both contracts and both statistics. Trading dates originate from `ts_ref`; publication timestamps control availability. No crossover is computed here. A publication-aware lookup may expose decision-session data if already available; this is distinct from ADV20, which must strictly exclude that session. Signals cannot affect the preceding return interval. Do not treat missing OI as a crossover or assume a one-session date shift proves availability.

In [ ]:
pair = [event["outgoing_contract"], event["baseline_destination"]]
source_date, aligned = f.observations.latest_common(
    pair, ["cleared_volume", "open_interest"], event_context)
assert source_date is not None
pprint([{"contract": c, "statistic": k, "trading_date": selected.observation.trading_date,
         "publication_timestamp": selected.observation.publication_timestamp,
         "value": selected.observation.statistic_value,
         "available": selected.observation.view(event_context)["available_as_of"]}
        for (c, k), selected in aligned.items()])


### Expected timing output and work to complete

Keep an unchanged outgoing/incoming pair with an explicit decision date, cut-off, next return-interval end and fallback flag. The example below is the shared baseline event, not a crossover result. Fields supplement the shared roll-event shape; they do not replace the production schema.

**TODO Lucie:** implement `build_baseline_roll_pairs`, `fixed_roll_date`, `lagged_volume_crossover`, `lagged_oi_crossover`, `apply_roll_timing_rule` and `build_roll_timing_portfolio`. Write hand tests for -5/-10 observed-session offsets, source-date/publication timing, missing OI, no-crossover baseline fallback, identical destination pairs and exit before eligibility ends. No-crossover fallback uses baseline -5 and is flagged; missing information must not create a signal. Open your own Part 2 issue-linked PR.

In [ ]:
baseline_timing_example = {
    **event, "decision_cutoff": event_context.cutoff_utc,
    "return_interval_end": next_return_date, "fallback_flag": False,
}
pprint(baseline_timing_example)
# TODO: invoke your own tested timing engine once available.
alternative_timing_output = None


### Integration boundary: future engine output → shared accounting

The TODO below is intentionally inactive. Your engine must produce canonical `Holding` rows for the baseline and all three alternatives on the same fixture dates, plus truthful decision-validity flags. Each row's `date` is the end of `(t-1,t]`; quantities and inputs are fixed by the preceding cut-off. Preserve exact contracts and explicit zero positions. Decision timestamps cannot be invented from later valuation marks.

Use the supplied fixture benchmark once and call the shared engine once for all compared methods. Missing required settlements invalidate the common date; never fill them with zero or splice maturities. One fixed AUM divides every portfolio's P&L. This adapter is for the synthetic accounting example only; the hand-authored benchmark quantities are not a constructed 60-session portfolio. Phase B must supply the actual common benchmark and frozen comparison sample.

In [ ]:
from src.portfolio.holdings import HoldingsBook
from src.metrics.portfolio_pnl import portfolio_pnl
from src.metrics.tracking_rmse_bps import tracking_rmse_bps

future_holding_rows = None  # TODO: baseline + three alternatives, produced by your engine.
future_decision_validity = None  # TODO: {(date, strategy): bool}, including the benchmark.
print("Canonical fields:", HOLDINGS_COLUMNS)
if future_holding_rows is None or future_decision_validity is None:
    print("TODO adapter: no strategy implemented or evaluated by this starter.")
else:
    benchmark_rows = [r for r in f.holdings.rows if r.strategy == f.raw["benchmark_strategy"]]
    strategies = [f.raw["benchmark_strategy"]] + sorted({r.strategy for r in future_holding_rows})
    assert len(strategies) == len(set(strategies)) == 5, "Provide baseline and three distinct alternatives."
    book = HoldingsBook(benchmark_rows + list(future_holding_rows), f.calendar, f.catalogue)
    daily = portfolio_pnl(book, f.observations, f.raw["return_dates"], strategies,
                          f.raw["fixed_aum"], f.raw["snapshot_end_exclusive"],
                          decision_validity=future_decision_validity)
    pprint(tracking_rmse_bps(daily))  # Synthetic arithmetic check; no preferred-method selection.
